In [2]:
import sys
print(sys.executable)

C:\Users\User\AppData\Local\Programs\Python\Python313\python.exe


In [3]:
import os
import numpy as np
import pandas as pd
base_path = r"C:\Users\User\Downloads\hacknew\project LazaruS\data\cleaned"

demographics = pd.read_csv(os.path.join(base_path, "cleaned_patient_demographics.csv"))
telemetry = pd.read_csv(os.path.join(base_path, "cleaned_telemetry_logs.csv"))
pharmacy = pd.read_csv(os.path.join(base_path, "cleaned_prescription_audit.csv"))



In [4]:
def hex_to_int(x):
    try:
        return int(x, 16)
    except:
        return np.nan

telemetry['heart_rate_decoded'] = telemetry['heart_rate_hex'].apply(hex_to_int)

In [5]:
telemetry.head()

,packet_id,ghost_id,room_id,heart_rate_hex,spO2,heart_rate_decoded
0,0,G-528,299,0x51,97.0,81
1,1,G-343,210,0x49,96.5,73
2,2,G-305,371,0x4d,96.0,77
3,3,G-163,457,0x48,95.0,72
4,4,G-108,204,0x3e,94.0,62


In [6]:
demographics.head()

,internal_id,ghost_id,parity_group,name,age
0,0,G-100,0,Patient_0,58
1,1,G-100,1,Patient_1,50
2,2,G-101,0,Patient_2,37
3,3,G-101,1,Patient_3,52
4,4,G-102,0,Patient_4,83


In [7]:
demographics.sort_values(by=['ghost_id'], ascending=[False])

,internal_id,ghost_id,parity_group,name,age
999,999,G-599,1,Patient_999,28
998,998,G-599,0,Patient_998,68
997,997,G-598,1,Patient_997,43
996,996,G-598,0,Patient_996,84
995,995,G-597,1,Patient_995,75
...,...,...,...,...,...
5,5,G-102,1,Patient_5,55
2,2,G-101,0,Patient_2,37
3,3,G-101,1,Patient_3,52
1,1,G-100,1,Patient_1,50


In [8]:
demographics.head(345)

,internal_id,ghost_id,parity_group,name,age
0,0,G-100,0,Patient_0,58
1,1,G-100,1,Patient_1,50
2,2,G-101,0,Patient_2,37
3,3,G-101,1,Patient_3,52
4,4,G-102,0,Patient_4,83
...,...,...,...,...,...
340,340,G-270,0,Patient_340,61
341,341,G-270,1,Patient_341,55
342,342,G-271,0,Patient_342,54
343,343,G-271,1,Patient_343,19


In [ ]:
from sklearn.preprocessing import StandardScaler

features = telemetry[['heart_rate_decoded', 'spO2']].copy()


features = features.fillna(features.mean())


scaler = StandardScaler()
X = scaler.fit_transform(features)

In [ ]:
telemetry.head()

In [ ]:
telemetry["ghost_id"].value_counts()

In [ ]:
data=telemetry

In [ ]:
df1=data

In [ ]:
df1.head()

In [ ]:


data = data.merge(
    demographics[['ghost_id', 'parity_group', 'age']],
    on='ghost_id',
    how='left'
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler


def adjust_heart_rate(row):
    age = row['age']
    hr = row['heart_rate_decoded']
   
    return hr + (50 - age) * 0.3

def adjust_spo2(row):
    age = row['age']
    spo2 = row['spO2']
   
    return spo2 + (50 - age) * 0.02

data['hr_adjusted'] = data.apply(adjust_heart_rate, axis=1)
data['spo2_adjusted'] = data.apply(adjust_spo2, axis=1)


def differentiate_duplicates(group):
    """
    For same ghost_id & same age, make each row slightly unique
    """
    group = group.copy().reset_index(drop=True)
    
    if len(group) > 1:
        for idx in range(len(group)):
            # Offset increases per duplicate row
            unique_offset = (idx + 1) * 0.85
            group.loc[idx, 'hr_adjusted'] += unique_offset
            group.loc[idx, 'spo2_adjusted'] -= unique_offset * 0.05
            
    return group


data = data.groupby(['ghost_id', 'age'], group_keys=False).apply(differentiate_duplicates)


def resolve_with_gmm(df):
    df = df.copy()

    if len(df) < 3:
        df['cluster'] = range(len(df))
        return df

    features = df[['hr_adjusted', 'spo2_adjusted', 'age', 'parity_group']].copy()
    features = features.fillna(features.mean())


    features['age_hr_interaction'] = features['hr_adjusted'] * (features['age'] / 50)

    features['unique_signal'] = np.arange(len(df)) * 0.01

    scaler = StandardScaler()
    X = scaler.fit_transform(features)

    
    lowest_bic = np.inf
    best_k = 1
    for k in range(1, min(5, len(df))):
        gmm = GaussianMixture(n_components=k, random_state=42)
        gmm.fit(X)
        bic = gmm.bic(X)
        if bic < lowest_bic:
            lowest_bic = bic
            best_k = k

    gmm = GaussianMixture(n_components=best_k, random_state=42)
    df['cluster'] = gmm.fit_predict(X)

    # Fallback if only one cluster
    if len(df['cluster'].unique()) == 1:
        df['cluster'] = pd.qcut(
            df['hr_adjusted'],
            q=2,
            labels=False,
            duplicates='drop'
        )

    return df

data = data.groupby('ghost_id', group_keys=False).apply(resolve_with_gmm)


data['patient_id'] = (
    data['ghost_id'].astype(str) + "_" +
    data['parity_group'].astype(str) + "_" +
    data['cluster'].astype(str)
)

In [ ]:

duplicate_columns = ['hr_adjusted', 'spo2_adjusted']


data_unique = data.drop_duplicates(subset=duplicate_columns, keep='first').reset_index(drop=True)



In [ ]:
data.head()

In [ ]:
data.duplicated()

In [ ]:

pharmacy = pharmacy.drop_duplicates(subset=['ghost_id'], keep='first')

df = data.merge(
    pharmacy,
    on='ghost_id',
    how='left'
)

In [ ]:
df.duplicated()

In [ ]:
def decrypt(text, shift):
    result = ""
    for char in str(text):
        if char.isalpha():
            result += chr((ord(char) - shift - 65) % 26 + 65)
        else:
            result += char
    return result

df['decrypted_med'] = df.apply(
    lambda row: decrypt(row['scrambled_med'], int(row['age']) % 26),
    axis=1
)

In [ ]:
dfnew=df

In [ ]:
dfnew.head()

In [ ]:
dfnew['spO2'] = dfnew['spO2'].interpolate(method='linear')
dfnew['heart_rate_decoded'] = dfnew['heart_rate_decoded'].interpolate(method='linear')
dfnew = dfnew[
    (dfnew['heart_rate_decoded'] >= 40) &
    (dfnew['heart_rate_decoded'] <= 180)
]

In [ ]:
dfnew.head()

In [ ]:
dfnew.duplicated()

In [ ]:
df1=dfnew

In [ ]:
df1.count()

In [ ]:
df2=df1

In [ ]:
df1.duplicated()

In [ ]:
print(df2.columns)

In [ ]:
df2.head()

In [ ]:

def validate_signal(df1):
    df1 = df1.copy()

   
    df1['hr_diff'] = df1['heart_rate_decoded'].diff().abs()

    
    df1['spo2_diff'] = df1['spO2'].diff().abs()

    df1['valid'] = (
        (df1['hr_diff'] < 40) &   
        (df1['spo2_diff'] < 5)    
    )

    return df1

df1 = df1.groupby('patient_id', group_keys=False).apply(validate_signal)


df1 = df1[df1['valid'] == True]

In [ ]:
df1.head()

In [ ]:
print(df1.columns)

In [ ]:

from scipy.interpolate import UnivariateSpline

def smooth_signals(df1):
    df1 = df1.copy()

    if len(df1) < 4:
        return df1

    x = np.arange(len(df1))

   
    spline_hr = UnivariateSpline(x, df1['heart_rate_decoded'], s=5)
    df1['hr_smooth'] = spline_hr(x)

  
    spline_spo2 = UnivariateSpline(x, df1['spO2'], s=2)
    df1['spo2_smooth'] = spline_spo2(x)

    return df1

df1 = df1.groupby('patient_id', group_keys=False).apply(smooth_signals)

In [ ]:
df1['heart_rate_final'] = df1['hr_smooth'].fillna(df1['heart_rate_decoded'])
df1['spo2_final'] = df1['spo2_smooth'].fillna(df1['spO2'])

In [ ]:
df1.head()

In [ ]:
df1.count()

In [ ]:
import matplotlib.pyplot as plt
plt.figure()

plt.plot(df1['heart_rate_decoded'].values[:50], label="Original HR")


plt.legend()
plt.title("Signal construction")
plt.show()

In [ ]:

LOW_HR = 60
HIGH_HR = 100


LOW_SPO2 = 92
CRITICAL_SPO2 = 88

def classify_heart_rate(hr):
    if hr < LOW_HR:
        return "LOW"
    elif hr > HIGH_HR:
        return "HIGH"
    else:
        return "NORMAL"

def classify_spo2(spo2):
    if spo2 < CRITICAL_SPO2:
        return "CRITICAL"
    elif spo2 < LOW_SPO2:
        return "LOW"
    else:
        return "NORMAL"

In [ ]:
print(df1.columns)

In [ ]:
df1['hr_status'] = df1['hr_adjusted'].apply(classify_heart_rate)
df1['spo2_status'] = df1['spo2_adjusted'].apply(classify_spo2)

In [ ]:
def overall_status(row):
    if row['spo2_status'] == "CRITICAL":
        return "CRITICAL"
    elif row['hr_status'] == "HIGH" or row['hr_status'] == "LOW":
        return "WARNING"
    else:
        return "STABLE"

df1['overall_status'] = df1.apply(overall_status, axis=1)

In [ ]:
print(df1.columns)

In [ ]:
df1 = df1.drop(columns=['name'], errors='ignore')

df1 = df1.merge(
    demographics[['ghost_id', 'name']],
    on='ghost_id',
    how='left'
)

In [ ]:
print(df1.columns)

In [ ]:
final_df = df1[[
    'patient_id',
    'name',
    'ghost_id',
    'age',
    'heart_rate_hex',        
    'heart_rate_decoded',      
    'hr_adjusted',
    'spo2_adjusted',
    'hr_status',
    'spo2_status',
    'overall_status'
]]

In [ ]:
final=final_df

In [ ]:
final_df=final_df.drop_duplicates(subset=['hr_adjusted','spo2_adjusted'],keep='first').reset_index(drop=True)

In [ ]:
final_df.duplicated()

In [ ]:
final_df.to_csv("trustmeicandoit_data.csv", index=False)

In [ ]:
import joblib

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(gmm, 'gmm_model.pkl')